# Download de XML de NF-e pelo Meu Danfe

Lê as chaves de acesso de um relatório do SEFAZ (ex.: **ACGRS103 – Rel Notas Fiscais Várias Origens**), consulta cada uma no [meudanfe.com.br](https://meudanfe.com.br/) e salva os XMLs numa pasta.

**Como funciona:** abre uma janela real do Chrome/Chromium e repete o que você faria à mão: cola a chave, clica em *Buscar* e depois em *Baixar XML*.

- O site tem uma verificação anti-robô (Cloudflare). Normalmente ela passa sozinha. Se aparecer um desafio na janela, **resolva você mesmo** e o script continua.
- Chaves já baixadas são puladas, então é só rodar de novo para completar o que falhou.
- No final, a célula 5 junta todos os XMLs numa planilha (cabeçalho + itens).

## 1. Instalação (rode só uma vez)

In [ ]:
%pip install -q playwright pandas openpyxl
!playwright install chromium

## 2. Configuração

In [ ]:
from pathlib import Path

PLANILHA    = r"ACGRS103 - Rel Notas Fiscais Varias Origens.xlsx"  # relatório com as chaves
COLUNA      = "CODG CHAVE NFE"                   # coluna com a chave de 44 dígitos
PASTA_XML   = Path(r"xml_nfe")                   # onde os XMLs serão salvos
PERFIL      = Path(r"perfil_navegador")          # guarda cookies entre execuções (ajuda na verificação do site)
INTERVALO   = (2.0, 4.0)                         # pausa aleatória entre consultas (segundos) — não sobrecarregar o site
TENTATIVAS  = 3                                  # tentativas por chave

## 3. Funções

In [ ]:
import asyncio, random, re, sys, time, concurrent.futures
from datetime import datetime
import pandas as pd
from playwright.async_api import async_playwright

URL = "https://meudanfe.com.br/"


def chave_valida(chave: str) -> bool:
    """44 dígitos + dígito verificador (módulo 11)."""
    if not re.fullmatch(r"\d{44}", chave):
        return False
    pesos = [2, 3, 4, 5, 6, 7, 8, 9]
    soma = sum(int(d) * pesos[i % 8] for i, d in enumerate(reversed(chave[:43])))
    dv = 11 - soma % 11
    return int(chave[43]) == (0 if dv >= 10 else dv)


def ler_chaves(planilha, coluna=COLUNA):
    """Lê a planilha e devolve as chaves únicas e válidas."""
    df = pd.read_excel(planilha, dtype=str)
    if coluna not in df.columns:
        raise KeyError(f"Coluna '{coluna}' não encontrada. Colunas disponíveis: {list(df.columns)}")
    chaves = df[coluna].dropna().str.replace(r"\D", "", regex=True)
    chaves = chaves[chaves != ""].drop_duplicates().tolist()
    invalidas = [c for c in chaves if not chave_valida(c)]
    if invalidas:
        print(f"⚠ {len(invalidas)} chave(s) inválida(s) ignorada(s):", *invalidas[:10], sep="\n  ")
    return [c for c in chaves if chave_valida(c)]


def arquivo_xml(chave):
    return PASTA_XML / f"NFE-{chave}.xml"


async def _baixar_uma(page, chave, timeout_s=25):
    """Consulta uma chave e salva o XML. Retorna (status, detalhe)."""
    await page.goto(URL, wait_until="domcontentloaded")
    await page.wait_for_timeout(2500)          # tempo para a verificação do site carregar
    await page.fill("#searchTxt", chave)
    await page.click("#searchBtn")

    botao  = page.locator("#downloadXmlBtn")
    alerta = page.locator("text=/Captcha|Falha ao consultar|não encontrad/i")
    fim = time.time() + timeout_s
    while time.time() < fim:
        if await botao.is_visible():
            async with page.expect_download(timeout=timeout_s * 1000) as dl:
                await botao.click()
            destino = arquivo_xml(chave)
            await (await dl.value).save_as(destino)
            return "OK", f"{destino.stat().st_size} bytes"
        if await alerta.count() and await alerta.first.is_visible():
            return "ERRO", (await alerta.first.inner_text()).strip()[:150]
        await page.wait_for_timeout(500)
    return "ERRO", "tempo esgotado (se apareceu um desafio na janela, resolva-o)"


async def _baixar_todas(chaves):
    PASTA_XML.mkdir(parents=True, exist_ok=True)
    pendentes = [c for c in chaves if not arquivo_xml(c).exists()]
    print(f"{len(chaves)} chaves | {len(chaves) - len(pendentes)} já baixadas | {len(pendentes)} a baixar\n")
    log = []
    async with async_playwright() as p:
        ctx = await p.chromium.launch_persistent_context(
            str(PERFIL), headless=False, accept_downloads=True)
        page = ctx.pages[0] if ctx.pages else await ctx.new_page()
        for i, chave in enumerate(pendentes, 1):
            for t in range(1, TENTATIVAS + 1):
                try:
                    status, detalhe = await _baixar_uma(page, chave)
                except Exception as e:
                    status, detalhe = "ERRO", f"{type(e).__name__}: {e}"[:150]
                if status == "OK":
                    break
                await asyncio.sleep(5 * t)     # espera mais a cada nova tentativa
            log.append({"chave": chave, "status": status, "detalhe": detalhe, "tentativas": t,
                        "quando": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})
            print(f"[{i}/{len(pendentes)}] {chave}  {status}  {detalhe}")
            await asyncio.sleep(random.uniform(*INTERVALO))
        await ctx.close()
    return pd.DataFrame(log, columns=["chave", "status", "detalhe", "tentativas", "quando"])


def baixar_xmls(chaves):
    """Roda o download numa thread própria (evita conflito com o loop do Jupyter, inclusive no Windows)."""
    def _run():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        try:
            return loop.run_until_complete(_baixar_todas(chaves))
        finally:
            loop.close()
    with concurrent.futures.ThreadPoolExecutor(1) as ex:
        return ex.submit(_run).result()

## 4. Baixar
Se algo falhar, rode a célula de novo: só as chaves que faltam serão consultadas.

In [ ]:
chaves = ler_chaves(PLANILHA)
log = baixar_xmls(chaves)

faltando = [c for c in chaves if not arquivo_xml(c).exists()]
print(f"\n✔ {len(chaves) - len(faltando)} de {len(chaves)} XMLs na pasta '{PASTA_XML}'.")
if faltando:
    print(f"✘ Faltam {len(faltando)}. Rode esta célula de novo.")
if not log.empty:
    log.to_csv(PASTA_XML / f"log_{datetime.now():%Y%m%d_%H%M%S}.csv", index=False, sep=";", encoding="utf-8-sig")
log[log.status != "OK"]

## 5. (Opcional) Juntar os XMLs numa planilha
Gera `consolidado_nfe.xlsx` com duas abas: **Notas** (uma linha por NF) e **Itens** (uma linha por produto).

In [ ]:
import xml.etree.ElementTree as ET

NS = {"n": "http://www.portalfiscal.inf.br/nfe"}

def _t(el, caminho):
    x = el.find(caminho, NS) if el is not None else None
    return x.text.strip() if x is not None and x.text else None

def _num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

def ler_xml(arquivo):
    raiz = ET.parse(arquivo).getroot()
    inf = raiz.find(".//n:infNFe", NS)
    if inf is None:
        raise ValueError("não é um XML de NF-e")
    chave = inf.get("Id", "").replace("NFe", "")
    ide, emit, dest = inf.find("n:ide", NS), inf.find("n:emit", NS), inf.find("n:dest", NS)
    tot = inf.find("n:total/n:ICMSTot", NS)
    nota = {
        "chave": chave,
        "numero": _t(ide, "n:nNF"), "serie": _t(ide, "n:serie"),
        "emissao": (_t(ide, "n:dhEmi") or _t(ide, "n:dEmi") or "")[:19].replace("T", " "),
        "natureza": _t(ide, "n:natOp"),
        "emit_cnpj": _t(emit, "n:CNPJ") or _t(emit, "n:CPF"), "emit_nome": _t(emit, "n:xNome"),
        "emit_municipio": _t(emit, "n:enderEmit/n:xMun"), "emit_uf": _t(emit, "n:enderEmit/n:UF"),
        "dest_cnpj": _t(dest, "n:CNPJ") or _t(dest, "n:CPF"), "dest_nome": _t(dest, "n:xNome"),
        "dest_municipio": _t(dest, "n:enderDest/n:xMun"),
        "valor_produtos": _num(_t(tot, "n:vProd")), "desconto": _num(_t(tot, "n:vDesc")),
        "frete": _num(_t(tot, "n:vFrete")), "valor_nf": _num(_t(tot, "n:vNF")),
        "protocolo": _t(raiz, ".//n:protNFe/n:infProt/n:nProt"),
        "info_complementar": _t(inf, "n:infAdic/n:infCpl"),
    }
    itens = []
    for det in inf.findall("n:det", NS):
        pr = det.find("n:prod", NS)
        itens.append({
            "chave": chave, "numero_nf": nota["numero"], "emit_nome": nota["emit_nome"],
            "item": det.get("nItem"), "codigo": _t(pr, "n:cProd"), "descricao": _t(pr, "n:xProd"),
            "ncm": _t(pr, "n:NCM"), "cfop": _t(pr, "n:CFOP"), "unidade": _t(pr, "n:uCom"),
            "quantidade": _num(_t(pr, "n:qCom")), "valor_unit": _num(_t(pr, "n:vUnCom")),
            "valor_total": _num(_t(pr, "n:vProd")),
        })
    return nota, itens

notas, itens, erros = [], [], []
for arq in sorted(PASTA_XML.glob("*.xml")):
    try:
        n, i = ler_xml(arq)
        notas.append(n); itens.extend(i)
    except Exception as e:
        erros.append((arq.name, str(e)))

df_notas, df_itens = pd.DataFrame(notas), pd.DataFrame(itens)
saida = PASTA_XML / "consolidado_nfe.xlsx"
with pd.ExcelWriter(saida) as w:
    df_notas.to_excel(w, sheet_name="Notas", index=False)
    df_itens.to_excel(w, sheet_name="Itens", index=False)

print(f"{len(df_notas)} notas, {len(df_itens)} itens → {saida}")
if erros:
    print("Arquivos com problema:", *erros, sep="\n  ")
df_notas.head()